# Random Codenet sample (Hugging Face)

Loads [KrishPS/codenet-accepted-c](https://huggingface.co/datasets/KrishPS/codenet-accepted-c), picks **5 distinct problems** (one fixed: `p00001`), and writes a CSV with **problem id**, **answer** (one accepted C submission), and **test cases** (JSON array string).

Requires: `pip install datasets pandas` and a network connection for the first run.

In [1]:
import json
import random

import pandas as pd
from datasets import load_dataset

DATASET = "KrishPS/codenet-accepted-c"
SPLIT = "train"
MUST_INCLUDE = "p00000"
N_PROBLEMS = 100
CSV_PATH = "codenet_random_100.csv"
RANDOM_SEED = 42  # change for a different random draw

random.seed(RANDOM_SEED)

print(f"Loading {DATASET} …")
ds = load_dataset(DATASET, split=SPLIT)
df = ds.to_pandas()

# One row per problem (first submission in dataset order; tests are identical per problem).
by_prob = df.drop_duplicates(subset=["problem_id"], keep="first").reset_index(drop=True)

fixed = by_prob[by_prob["problem_id"] == MUST_INCLUDE]
if fixed.empty:
    raise ValueError(f"{MUST_INCLUDE!r} not found in dataset — check problem_id spelling.")

pool = by_prob[by_prob["problem_id"] != MUST_INCLUDE]
k = N_PROBLEMS - 1
if len(pool) < k:
    raise ValueError(f"Need {k} other problems but pool has only {len(pool)}")

sample_idx = pool.sample(n=k, random_state=RANDOM_SEED).index.tolist()
out = pd.concat([fixed, pool.loc[sample_idx]], ignore_index=True)

export = pd.DataFrame(
    {
        "problem_id": out["problem_id"],
        "answer": out["code"],
        "test_cases": out["test_cases"],
        "num_test_cases": out["num_test_cases"],
    }
)

export.to_csv(CSV_PATH, index=False, encoding="utf-8")
print(f"Wrote {len(export)} rows → {CSV_PATH}")
print(export[["problem_id", "num_test_cases"]])

# Quick peek: first test case of p00001
row0 = export[export["problem_id"] == MUST_INCLUDE].iloc[0]
tc0 = json.loads(row0["test_cases"])[0]
print(f"\n{MUST_INCLUDE} — first test input (first 200 chars):", repr(tc0.get("input", "")[:200]))

/Users/shreybirmiwal/projects/jwlabs/gen-optimize-assembly/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading KrishPS/codenet-accepted-c …
Wrote 100 rows → codenet_random_100.csv
   problem_id  num_test_cases
0      p00000               1
1      p02283             102
2      p00135             102
3      p00446             102
4      p00204             102
..        ...             ...
95     p00329             104
96     p01715               1
97     p03482               1
98     p01560             104
99     p00367               1

[100 rows x 2 columns]

p00000 — first test input (first 200 chars): '\n'
